# Checa AI - BERTimbau para risco textual no WhatsApp
Este notebook treina um sinal auxiliar de semelhança textual. A saída não é probabilidade de falsidade e não substitui verificação por evidências.

In [ ]:
!pip -q install 'transformers==4.57.6' 'accelerate>=1.2' 'scikit-learn>=1.6,<2' pandas safetensors
!git clone -q --depth 1 https://github.com/cabrau/FakeWhatsApp.Br.git /content/FakeWhatsApp.Br || true

In [ ]:
import hashlib, json, re, unicodedata
from pathlib import Path
import pandas as pd

SOURCE = Path('/content/FakeWhatsApp.Br/data/2018/fakeWhatsApp.BR_2018.csv')
df = pd.read_csv(SOURCE, low_memory=False)
df = df[df.misinformation.isin([0, 1])].copy()
df = df[(df.midia == 0) & df.text.notna() & (df.text.str.split().str.len() >= 3) & (df.text.str.len() <= 10000)]
df['timestamp'] = pd.to_datetime(df.timestamp)
url_re = re.compile(r'https?://\S+|www\.\S+', re.I)
def norm(text):
    text = unicodedata.normalize('NFKC', text).casefold()
    return re.sub(r'\s+', ' ', url_re.sub(' URL ', text)).strip()
df['group_id'] = df.text.map(lambda x: hashlib.sha256(norm(x).encode()).hexdigest()[:20])
df['split'] = 'gap'
df.loc[df.timestamp.dt.date <= pd.Timestamp('2018-08-31').date(), 'split'] = 'train'
df.loc[(df.timestamp.dt.date >= pd.Timestamp('2018-09-01').date()) & (df.timestamp.dt.date <= pd.Timestamp('2018-09-30').date()), 'split'] = 'validation'
df.loc[df.timestamp.dt.date >= pd.Timestamp('2018-10-01').date(), 'split'] = 'test'
valid_groups = df.groupby('group_id').filter(lambda g: g.misinformation.nunique() == 1 and g.split.nunique() == 1)
dataset = valid_groups.sort_values('timestamp').drop_duplicates('group_id')[['text','misinformation','split','group_id']]
print(dataset.groupby(['split','misinformation']).size())
assert dataset.groupby('group_id').split.nunique().max() == 1

In [ ]:
import numpy as np, torch
from sklearn.metrics import accuracy_score, average_precision_score, brier_score_loss, f1_score, precision_score, recall_score, roc_auc_score
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, EarlyStoppingCallback, Trainer, TrainingArguments

MODEL = 'neuralmind/bert-base-portuguese-cased'
# Commit mais recente do repositório do modelo em 2026-10-14.
REVISION = '94d69c9'
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION, use_fast=True)
class Messages(Dataset):
    def __init__(self, frame):
        self.labels = frame.misinformation.astype(int).tolist()
        self.enc = tokenizer(frame.text.tolist(), truncation=True, max_length=128, padding=False)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        row = {k: v[i] for k, v in self.enc.items()}; row['labels'] = self.labels[i]; return row
sets = {s: Messages(dataset[dataset.split == s]) for s in ['train','validation','test']}
def metrics(pred):
    probs = torch.softmax(torch.tensor(pred.predictions), dim=1).numpy()[:,1]
    y = pred.label_ids; yhat = (probs >= .5).astype(int)
    return {'accuracy': accuracy_score(y,yhat), 'f1_macro': f1_score(y,yhat,average='macro'), 'f1_misinformation': f1_score(y,yhat), 'precision_misinformation': precision_score(y,yhat), 'recall_misinformation': recall_score(y,yhat), 'roc_auc': roc_auc_score(y,probs), 'pr_auc': average_precision_score(y,probs), 'brier': brier_score_loss(y,probs)}
model = AutoModelForSequenceClassification.from_pretrained(MODEL, revision=REVISION, num_labels=2, id2label={0:'NON_MISINFORMATION',1:'MISINFORMATION'}, label2id={'NON_MISINFORMATION':0,'MISINFORMATION':1})
args = TrainingArguments(output_dir='/content/whatsapp-risk-runs', eval_strategy='epoch', save_strategy='epoch', learning_rate=2e-5, per_device_train_batch_size=16, per_device_eval_batch_size=32, num_train_epochs=3, weight_decay=.01, warmup_ratio=.1, load_best_model_at_end=True, metric_for_best_model='f1_macro', greater_is_better=True, save_total_limit=1, fp16=torch.cuda.is_available(), report_to=[], seed=42, data_seed=42)
trainer = Trainer(model=model, args=args, train_dataset=sets['train'], eval_dataset=sets['validation'], processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'WARNING: CPU runtime')
trainer.train()

In [ ]:
from google.colab import files
out = Path('/content/whatsapp-risk-bertimbau')
trainer.save_model(out); tokenizer.save_pretrained(out)
test_output = trainer.predict(sets['test'], metric_key_prefix='test')
result = {'task':'whatsapp_textual_misinformation_risk','model':MODEL,'revision':REVISION,'max_length':128,'train':len(sets['train']),'validation':len(sets['validation']),'test':len(sets['test']),'metrics':{k:float(v) for k,v in test_output.metrics.items()}}
(out/'metrics.json').write_text(json.dumps(result,ensure_ascii=False,indent=2),encoding='utf-8')
!cd /content && zip -qr whatsapp-risk-bertimbau.zip whatsapp-risk-bertimbau
print(json.dumps(result, indent=2, ensure_ascii=False))
files.download('/content/whatsapp-risk-bertimbau.zip')